# PARA-1a 部A 型分け（CR-1a のデータ・CPU・API なし）

凍結した PARA-1 の部A を、PARA-1a の修正のとおり CR-1a のデータで行います（CR-1a は `INSUFFICIENT_FORK_SIGNAL` だったので、決まりどおり部A は行います）。

- **セル1**：決まった版のコードを取ってきて、NLTK（凍結のときと同じコミット）と WordNet・stopwords（凍結の記録と SHA256 が一致するもの）を入れます。違えば止まります。
- **セル2（G0）**：G0a（付録の 45 組がすべて期待の型）→ G0c（人工の世界 200 個で、ずらした比較の偽陽性の割合）。G0c がいちばん時間がかかります（15〜30 分くらい）。途中で切れても、Drive の途中経過から続きをやります。
- **セル3（部A）**：G1・G2 の非公開ファイルをハッシュで確かめてから、置き換えを取り出して A1〜A3 と副次 A4〜A7、判定の順でラベルを出します。
- 問題文・応答・語の中身はどこにも表示しません。結果は数・ラベル・番号だけで、Drive の `DECISION_POINTS/cr1a/PARA1A/` に入ります。
- 費用 $0。シークレットは `GITHUB_TOKEN` だけ。CPU のランタイムで、セル1 → セル2 → セル3 の順に（「すべてのセルを実行」で OK）。全部で 30〜45 分くらい。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, pinned NLTK code and data, notifications.  No API key.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, urllib.request, zipfile
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
STUDY = "CR-1a"
EXPECTED_HEAD = "e0ac94a0cc0b5033223285e88d9b9e3590f50025"
G1_HEAD = "7d95ee90a7b536fb85326e2a30981b9bc8030421"
G2_HEAD = "f6c6bb94fe2ff77a5726e71b835f7ee0632a9d4d"
G1_ITEMS_SHA = "bd42639245cb234f86e0bcd97092db67ae2842c9f1e48e123a9ee1b484c931b5"
G2_SUMMARY_SHA = "17afa043db733289e49eb4d55ac1c160760067f36eb2a9c4cdfa36e956f159f1"
G3_RESULT_SHA = "4addb0ed40b914547e272b1c450285425ea9cf8c76f81b1c28576b127c8217a7"
PARA1_FREEZE, PARA1A_FREEZE, LOCK3 = "69dd041", "7d95ee9", "69b43d8"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
def is_anc(a, b):
    return subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", a, b]).returncode == 0
for anc in (PARA1_FREEZE, PARA1A_FREEZE, LOCK3, G1_HEAD, G2_HEAD, "bcafae7", "bbd6b4a"):
    assert is_anc(anc, head), anc + " が祖先ではありません"
TIMING = {"para1_freeze": PARA1_FREEZE, "para1a_freeze": PARA1A_FREEZE, "lock3": LOCK3,
          "para1_freeze_is_ancestor_of_lock3": is_anc(PARA1_FREEZE, LOCK3),
          "para1a_freeze_is_ancestor_of_lock3": is_anc(PARA1A_FREEZE, LOCK3)}
assert TIMING["para1_freeze_is_ancestor_of_lock3"] and TIMING["para1a_freeze_is_ancestor_of_lock3"]
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
from pork import para1a_parta as P

# NLTK code: the commit used before the freeze.  Data: nltk_data gh-pages at the recorded commit, SHA256-checked.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                f"nltk @ git+https://github.com/nltk/nltk@{P.NLTK_COMMIT}"], check=True)
NLTK_DIR = Path("/content/nltk_data_pinned")
(NLTK_DIR / "corpora").mkdir(parents=True, exist_ok=True)
for name, want in (("wordnet", P.WORDNET_ZIP_SHA), ("stopwords", P.STOPWORDS_ZIP_SHA)):
    z = NLTK_DIR / "corpora" / (name + ".zip")
    url = f"https://raw.githubusercontent.com/nltk/nltk_data/{P.NLTK_DATA_COMMIT}/packages/corpora/{name}.zip"
    z.write_bytes(urllib.request.urlopen(url, timeout=120).read())
    got = hashlib.sha256(z.read_bytes()).hexdigest()
    assert got == want, name + ".zip の SHA256 が凍結の記録と違います。止めます。"
    shutil.rmtree(NLTK_DIR / "corpora" / name, ignore_errors=True)
    zipfile.ZipFile(z).extractall(NLTK_DIR / "corpora")
os.environ["NLTK_DATA"] = str(NLTK_DIR)
import nltk
nltk.data.path[:] = [str(NLTK_DIR)]
from nltk.corpus import wordnet as wn, stopwords
STOP = stopwords.words("english")
assert hashlib.sha256((NLTK_DIR / "corpora/stopwords/english").read_bytes()).hexdigest() == P.STOPWORDS_EN_FILE_SHA
assert len(STOP) == 198 and hashlib.sha256("\n".join(STOP).encode("utf-8")).hexdigest() == P.STOPWORDS_LIST_SHA
assert wn.get_version() == "3.0"
import numpy
VERSIONS = {"nltk": nltk.__version__, "nltk_commit": P.NLTK_COMMIT, "nltk_data_commit": P.NLTK_DATA_COMMIT,
            "wordnet": wn.get_version(), "stopwords_en": len(STOP), "numpy": numpy.__version__,
            "python": sys.version.split()[0], "cpus": os.cpu_count()}
print("exact head:", head, "|", VERSIONS)
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a/PARA1A")
OUT.mkdir(parents=True, exist_ok=True)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'PARA-1a 部A 型分け（CR-1a・CPU・API なし）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - PARA-1a Part A typing', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — G0: G0a (45 pairs) and G0c (200 artificial worlds).  No benchmark data is read here.
t0 = time.time()
TYPING_PY = REPO / "scripts/para1_typing.py"
mod_main = P.load_typing_module(TYPING_PY, "para1_typing_main")
g0a = P.run_g0a(mod_main, wn, STOP)
print("G0a:", g0a["pairs"], "組中", g0a["pairs"] - g0a["mismatches"], "組が一致")
assert g0a["pass"], "G0a が通りません。データに進みません。"
T = P.Typing(mod_main, wn, STOP)
table, table_sha = P.build_word_table(REPO / "scripts/para1_g0c_wordtable.py", wn, STOP)
print("G0c の語の表:", {k: len(v) for k, v in table}, "| SHA256 一致")
G0C_PART = OUT / "PARA1A_G0c_worlds_partial.json"
key = {"code_head": head, "table_sha256": table_sha, "n_shuffle": P.N_SHUFFLE}
done = []
if G0C_PART.exists():
    prev = json.loads(G0C_PART.read_text())
    if prev.get("key") == key:
        done = prev["worlds"]
        print("途中経過から再開します：", len(done), "世界ずみ")
def save_worlds(ws):
    tmp = G0C_PART.with_name(G0C_PART.name + ".tmp")
    tmp.write_text(json.dumps({"key": key, "worlds": ws}, ensure_ascii=False))
    os.replace(tmp, G0C_PART)
    print(f"  G0c: {len(ws)}/{P.N_WORLDS} 世界 | {round(time.time() - t0)} 秒")
worlds = P.run_g0c(T, table, done=done, save=save_worlds)
G0 = {"G0a": g0a, "G0c": P.g0c_summary(worlds, table_sha), "versions": VERSIONS,
      "typing_blob": P.TYPING_BLOB, "wordtable_blob": P.WORDTABLE_BLOB, "code_head": head,
      "seconds": round(time.time() - t0, 1)}
(OUT / "PARA1A_G0_result.json").write_text(json.dumps(G0, ensure_ascii=False, indent=1))
print("G0c 偽陽性の割合:", G0["G0c"]["false_positive_share"], "| 通過:", G0["G0c"]["pass"], "| 時間:", G0["seconds"], "秒")

In [ ]:
# Cell 3 — Part A on the CR-1a data.  Integrity checks first; numbers / labels / ids only.
t1 = time.time()
from pork import cr1_run as R
from pork import cr1_requests as RQ
from pork import cr1_analysis as A
from pork import cr1_g3 as G3
from pork import cr1_items as CI
g3_text = (REPO / "reports/cr1a_g3_result.json").read_text()
assert hashlib.sha256(g3_text.encode("utf-8")).hexdigest() == G3_RESULT_SHA, "G3 の結果ファイルが違います。止めます。"
g3 = json.loads(g3_text)
CR1A_VERDICT = g3["verdict"]
rows, DATA_ID = R.load_rows_public()
chain = CI.chain_and_candidates(rows)
del rows
assert CI.candidates_sha256(chain) == R.CANDIDATES_SHA256, "候補の一覧が段2 と違います。止めます。"
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a")
ck = json.loads((D / "G1/CR1A_G1_checkpoint_PRIVATE.json").read_text())
assert ck["header"] == R.header(STUDY, G1_HEAD, R.CANDIDATES_SHA256, A.MODEL, RQ.SDK_VERSION)
R.validate(ck, chain)
g1 = R.summarize_g1(ck)
assert g1["verdict"] == "GO" and g1["final_items_ids_sha256"] == G1_ITEMS_SHA
s2_text = (D / "G2/CR1A_G2_summary.json").read_text()
assert hashlib.sha256(s2_text.encode("utf-8")).hexdigest() == G2_SUMMARY_SHA, "G2 の記録が変わっています。止めます。"
g2 = json.loads(s2_text)
assert g2["done"] and g2["code_head"] == G2_HEAD
TIMING["cr1a_first_submit"] = g2["first_submit"]
final = g1["final_items"]
res, load_stats = G3.load_results(D / "G2", g2["batch_records"], final)
print("読み込み:", load_stats["requests"], "件 | 状態:", load_stats["final_status"])
by_item = G3.per_item(res, len(final))
order = P.item_order(final)
items, n_missing = [], 0
for i, fi in enumerate(final):
    p = ck["items"][str(fi["template"])]
    assert hashlib.sha256(p["text"].encode("utf-8")).hexdigest() == fi["text_sha256"], f"item {i}: 土台の道が G1 と違います"
    built = A.build_item(i, int(fi["template"]), p["text"], by_item[i])
    if built.missing.any():
        n_missing += 1
        continue
    items.append(P.extract_item(i, int(fi["template"]), order[i], p["text"], by_item[i], u=built.u))
assert [it.index for it in items] == g3["main_item_indices"], "主の分析の問題が G3 と違います。止めます。"
del res, by_item
out = {"study": "PARA-1a Part A", "source": "CR-1a", "cr1a_verdict": CR1A_VERDICT,
       "para1_original_part_a": "NOT_RUN_CR1_STOPPED", "timing": TIMING, "G0": G0,
       "readings_fixed_before_data": ["R1 A2 positions = cuts with >= 1 group B substitution (all-cuts reading reported unlabeled)",
                                      "R2 A4 bin = min(9, 10(d-1)//L), d = L - cut, bin 0 nearest the answer",
                                      "R3 A6 strips whitespace only (case and marks kept)",
                                      "R4 G0c random order: per cut w first, then per continuation random() then the substitute",
                                      "R5 SEMANTIC=意味型 FORM=形型 BOTH=両方 DIGIT=数字(数字・形+数字・その他)"],
       "inputs": {"code_head": head, "g1_code_head": G1_HEAD, "g2_code_head": G2_HEAD, "g3_result_sha256": G3_RESULT_SHA,
                  "dataset": DATA_ID, "items_main": len(items), "items_missing": n_missing}}
if CR1A_VERDICT not in ("INSUFFICIENT_FORK_SIGNAL", "LABELS_ISSUED"):
    out.update(P.final_labels(CR1A_VERDICT, None))
else:
    T_nosim = P.Typing(P.load_typing_module(TYPING_PY, "para1_typing_nosim", similar_to=False), wn, STOP)
    T_raw = P.Typing(P.load_typing_module(TYPING_PY, "para1_typing_raw", keep_case_marks=True), wn, STOP)
    import numpy as np
    rng = np.random.default_rng([P.SEED_PARA_AUDIT, 2])
    pairs = [(it.a[j], it.words[c]) for it in items for j, c in enumerate(it.cut) if it.groupB[j]]
    pairs += [(it.a[j], it.words[int(rng.integers(it.L))]) for it in items for j in range(len(it.a)) if it.groupB[j]]
    audit = P.run_typing_audit(T, pairs)
    del pairs
    assert audit["pass"], "型分けの確かめ（速くした型分け vs 凍結のままの型分け）でずれが出ました。止めます。"
    part = P.analyze(items, T, T_nosim, T_raw, g0c_pass=G0["G0c"]["pass"])
    out.update(P.final_labels(CR1A_VERDICT, part))
    out["typing_audit"] = audit
    out["part_a"] = part
out["seconds"] = round(time.time() - t1, 1)
text = A.result_summary(out)
tmp = OUT / "PARA1A_partA_result.json.tmp"
tmp.write_text(text)
os.replace(tmp, OUT / "PARA1A_partA_result.json")
print("判定:", out["verdict"])
for lab in out["labels"]:
    print("  ", lab)
if "part_a" in out:
    print("置き換え（組B）:", out["part_a"]["A1"]["substitutions"], "本 | 問題:", out["part_a"]["A1"]["items"],
          "| ずらした比較の問題:", out["part_a"]["A2"]["items_compared"], "| 時間:", out["seconds"], "秒")
notify('終わりました：PARA-1a 部A の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()